In [1]:
import sys
sys.path.append('..')
import numpy as np
import pandas as pd
import torch
from scipy.stats import pearsonr, spearmanr
import tqdm.notebook as tqdm

import boda
from genoml.utils import *

In [2]:
malinois_path = '../pretrained_models/malinois/malinois_artifacts__20211113_021200__287348.tar'
my_model = boda.common.utils.load_model(malinois_path)

archive unpacked in ./


Loaded model from 20211113_021200 in eval mode


In [3]:
input_len = torch.load('./artifacts/torch_checkpoint.pt')['model_hparams'].input_len
input_len

600

In [4]:
left_pad_len = (input_len - 200) // 2
right_pad_len= (input_len - 200) - left_pad_len

left_flank = boda.common.utils.dna2tensor( 
    boda.common.constants.MPRA_UPSTREAM[-left_pad_len:] 
).unsqueeze(0)
print(f'left flank shape: {left_flank.shape}')

right_flank= boda.common.utils.dna2tensor( 
    boda.common.constants.MPRA_DOWNSTREAM[:right_pad_len] 
).unsqueeze(0)
right_flank.shape
print(f'right flank shape: {right_flank.shape}')

flank_builder = boda.common.utils.FlankBuilder(
    left_flank=left_flank,
    right_flank=right_flank,
)

flank_builder.cuda()

left flank shape: torch.Size([1, 4, 200])
right flank shape: torch.Size([1, 4, 200])


FlankBuilder()

In [6]:
mpra_19 = pd.read_table('../data/Gosai_MPRA/41586_2024_8070_MOESM4_ESM.txt', sep='\t', header=0)
mpra_19 = mpra_19.loc[ mpra_19.loc[:, ['K562_lfcSE', 'HepG2_lfcSE', 'SKNSH_lfcSE']].max(axis=1) < 1.0 ]
mpra_19

/tmp/ipykernel_84775/2365665045.py:1: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  mpra_19 = pd.read_table('../data/Gosai_MPRA/41586_2024_8070_MOESM4_ESM.txt', sep='\t', header=0)


,IDs,chr,data_project,OL,class,K562_log2FC,HepG2_log2FC,SKNSH_log2FC,K562_lfcSE,HepG2_lfcSE,SKNSH_lfcSE,sequence
0,7:70038969:G:T:A:wC,7,UKBB,29,"BMI,BFP",0.061,0.234,0.047,0.099,0.118,0.131,CCTGGTCTTTCTTGCTAAATAAACATATCGTGCATCATCCAGATCT...
1,1:192696196:C:T:A:wC,1,UKBB,33,Depression_GP,0.380,0.005,-0.244,0.162,0.186,0.119,CATAAAGATGAGGCTTGGCAAAGAACATCTCTCGGTGCCTCCCATT...
2,1:211209457:C:T:A:wC,1,UKBB,33,CAD,0.037,0.385,-0.005,0.098,0.122,0.087,CATAAAGCCAATCACTGAGATGACAAGTACTGCCAGGAAAGAAGGC...
3,15:89574440:GT:G:A:wC,15,UKBB,33,CAD,4.465,4.107,2.870,0.114,0.140,0.163,CATAAAGGCAGTGTAGACCCAAACAGTGAGCAGTAGCAAGATTTAT...
4,15:89574440:GT:G:R:wC,15,UKBB,33,CAD,4.509,4.116,3.040,0.157,0.209,0.195,CATAAAGGCAGTGTAGACCCAAACAGTGAGCAGTAGCAAGATTTAT...
...,...,...,...,...,...,...,...,...,...,...,...,...
798059,4:44680358:NA:NA,4,CRE,15,K27_All,7.444,5.344,6.585,0.098,0.070,0.141,CAGTAGTAAGAAAGAGACAATGCAAAGGAATTGGCACAGCACTCAG...
798060,18:9125893:NA:NA,18,CRE,15,K27_Uniq,-0.205,-0.157,-0.209,0.133,0.157,0.185,CAGTACTGCTGGCCCCAGAAAAGCCCCTCTCCTTATACCCTAGGCC...
798061,12:33905808:NA:NA,12,CRE,15,K27_Uniq,1.218,0.614,0.570,0.127,0.167,0.191,CAGTACCTTGTCCCCACTTCCCATTTGGCCTCTGGCAGAGGAGGAG...
798062,3:128145854:NA:NA,3,CRE,15,K27_Uniq,-0.222,-0.339,-0.818,0.159,0.198,0.239,CAGTACACCCCAGCTTCCAAAGGCCTTCTGTGACAAAGAGAGACTA...


In [7]:
pass_seq = mpra_19.loc[ mpra_19['sequence'].str.len() == 200 ].reset_index(drop=True)

seq_tensor  = torch.stack([ boda.common.utils.dna2tensor(x['sequence']) for i, x in tqdm.tqdm(pass_seq.iterrows(), total=pass_seq.shape[0]) ], dim=0)
seq_dataset = torch.utils.data.TensorDataset(seq_tensor)
seq_loader  = torch.utils.data.DataLoader(seq_dataset, batch_size=128)

  0%|          | 0/717741 [00:00<?, ?it/s]

In [8]:
results = []

with torch.no_grad():
    for i, batch in enumerate(tqdm.tqdm(seq_loader)):
        prepped_seq = flank_builder( batch[0].cuda() )
        predictions = my_model( prepped_seq ) + \
                      my_model( prepped_seq.flip(dims=[1,2]) ) # Also
        predictions = predictions.div(2.)
        results.append(predictions.detach().cpu())
                
predictions = torch.cat(results, dim=0)

  0%|          | 0/5608 [00:00<?, ?it/s]

In [9]:
pred_df     = pd.DataFrame( predictions.numpy(), columns=['K562_preds', 'HepG2_preds', 'SKNSH_preds'] )
all_results = pd.concat([pass_seq, pred_df], axis=1)
all_results

,IDs,chr,data_project,OL,class,K562_log2FC,HepG2_log2FC,SKNSH_log2FC,K562_lfcSE,HepG2_lfcSE,SKNSH_lfcSE,sequence,K562_preds,HepG2_preds,SKNSH_preds
0,7:70038969:G:T:A:wC,7,UKBB,29,"BMI,BFP",0.061,0.234,0.047,0.099,0.118,0.131,CCTGGTCTTTCTTGCTAAATAAACATATCGTGCATCATCCAGATCT...,0.023,0.492,0.471
1,1:192696196:C:T:A:wC,1,UKBB,33,Depression_GP,0.380,0.005,-0.244,0.162,0.186,0.119,CATAAAGATGAGGCTTGGCAAAGAACATCTCTCGGTGCCTCCCATT...,-0.148,-0.183,-0.357
2,1:211209457:C:T:A:wC,1,UKBB,33,CAD,0.037,0.385,-0.005,0.098,0.122,0.087,CATAAAGCCAATCACTGAGATGACAAGTACTGCCAGGAAAGAAGGC...,-0.171,0.196,-0.020
3,15:89574440:GT:G:R:wC,15,UKBB,33,CAD,4.509,4.116,3.040,0.157,0.209,0.195,CATAAAGGCAGTGTAGACCCAAACAGTGAGCAGTAGCAAGATTTAT...,4.553,3.783,4.055
4,12:63513920:G:A:A:wC,12,UKBB,32,Morning_Person,1.617,1.423,1.336,0.160,0.148,0.225,CATAAAGGGCTGAACATGCTGTTGAAAAAATGTAGATATAAAAGTT...,1.277,1.127,1.074
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
717736,4:44680358:NA:NA,4,CRE,15,K27_All,7.444,5.344,6.585,0.098,0.070,0.141,CAGTAGTAAGAAAGAGACAATGCAAAGGAATTGGCACAGCACTCAG...,4.824,4.612,5.403
717737,18:9125893:NA:NA,18,CRE,15,K27_Uniq,-0.205,-0.157,-0.209,0.133,0.157,0.185,CAGTACTGCTGGCCCCAGAAAAGCCCCTCTCCTTATACCCTAGGCC...,-0.097,0.148,-0.033
717738,12:33905808:NA:NA,12,CRE,15,K27_Uniq,1.218,0.614,0.570,0.127,0.167,0.191,CAGTACCTTGTCCCCACTTCCCATTTGGCCTCTGGCAGAGGAGGAG...,1.396,0.743,0.537
717739,3:128145854:NA:NA,3,CRE,15,K27_Uniq,-0.222,-0.339,-0.818,0.159,0.198,0.239,CAGTACACCCCAGCTTCCAAAGGCCTTCTGTGACAAAGAGAGACTA...,-0.097,-0.072,-0.321


In [10]:
chr_filter = (all_results['chr'] == 19) | \
             (all_results['chr'] == 21) | \
             (all_results['chr'] == '19') | \
             (all_results['chr'] == '21') | \
             (all_results['chr'] == 'X')

val_results = all_results.loc[ chr_filter ]

for cell in ['K562', 'HepG2', 'SKNSH']:
    corr = pearsonr(val_results[f'{cell}_log2FC'], val_results[f'{cell}_preds'])
    print(cell)
    print(f'stat: {corr[0]:.4f}, pvalue: {corr[1]}')

for cell in ['K562', 'HepG2', 'SKNSH']:
    corr = spearmanr(val_results[f'{cell}_log2FC'], val_results[f'{cell}_preds'])
    print(cell)
    print(f'stat: {corr[0]:.4f}, pvalue: {corr[1]}')

K562
stat: 0.9131, pvalue: 0.0
HepG2
stat: 0.9110, pvalue: 0.0
SKNSH
stat: 0.9073, pvalue: 0.0
K562
stat: 0.8405, pvalue: 0.0
HepG2
stat: 0.8615, pvalue: 0.0
SKNSH
stat: 0.8588, pvalue: 0.0


In [11]:
chr_filter = (all_results['chr'] == 7) | \
             (all_results['chr'] == 13) | \
             (all_results['chr'] == '7') | \
             (all_results['chr'] == '13')

test_results = all_results.loc[ chr_filter ]



for cell in ['K562', 'HepG2', 'SKNSH']:
    corr = pearsonr(test_results[f'{cell}_log2FC'], test_results[f'{cell}_preds'])
    print(cell)
    print(f'stat: {corr[0]:.4f}, pvalue: {corr[1]}')


test_results = all_results.loc[ chr_filter ]

for cell in ['K562', 'HepG2', 'SKNSH']:
    corr = spearmanr(test_results[f'{cell}_log2FC'], test_results[f'{cell}_preds'])
    print(cell)
    print(f'stat: {corr[0]:.4f}, pvalue: {corr[1]}')

K562
stat: 0.8842, pvalue: 0.0
HepG2
stat: 0.8880, pvalue: 0.0
SKNSH
stat: 0.8785, pvalue: 0.0
K562
stat: 0.8104, pvalue: 0.0
HepG2
stat: 0.8334, pvalue: 0.0
SKNSH
stat: 0.8306, pvalue: 0.0
